In [1]:
import pandas as pd
import bs4 as bs
import re
from group import RateGroup, RateAgreement, RateSteps
import json
from dateutil import parser
from datetime import datetime, timedelta
import html5lib

In [2]:
classDf = pd.read_pickle('classDf.pkl')
idContentDf = pd.read_pickle('idContentDf.pkl')

In [36]:
def getContentSoup(tbsId):
    ac = idContentDf.loc[idContentDf['ids'] == tbsId, 'htmlContent'].values[0].decode('utf-8', 'ignore')
    #print(ac)
    ac = ac.replace('“', '')
    ac = ac.replace('”', '|')
    ac = ac.replace(u'\xa0', ' ') #non breaking space
    ac = ac.replace('&#160;&#8211; ', '-') #non breaking space
    ac = ac.replace('&#8211; ', '-') #non breaking space
    
    soup = bs.BeautifulSoup(ac,'lxml')

    return soup

In [4]:
def getRateTables(tbsId, appendixText):
    soup = getContentSoup(tbsId)
    tables = soup.select('h2:-soup-contains("%s")' %(appendixText))[0].parent.findAll('table')
    return tables

In [137]:
def parseHTMLToJSON(tbsId, rateTables, sep, secondary, signingDate):
    rates = []
    previousGrp = ''
    for rateCat in rateTables:
        steps = rateCat.select('thead th')

        if len(steps) == 0:
            steps = rateCat.select('table:not(tfoot) > tr')[0].select('th')

        agreementTable = rateCat.select('tbody tr')
        if len(agreementTable) == 0:
            agreementTable = rateCat.select('table:not(tfoot) > tr')
            agreementTable.pop(0)

        rateAgreements = getAgreements(agreementTable, steps, signingDate)

        withSepGrpLvl = re.split('annual', rateCat.find('caption').getText(), flags=re.IGNORECASE)
        grpLvl = withSepGrpLvl[0].strip().rstrip(sep)
        grpLvl = grpLvl.rstrip('-').strip() #id=15 has some weird hyphens

        if len(grpLvl.rsplit('-',1)) == 1:
            if len(grpLvl.rsplit(sep,1)) == 1:
                grp = previousGrp #instead of knowing the group we store the previous group since that is what this would be
                lvl = grpLvl
            else:
                grp = grpLvl.rsplit(sep,1)[0].strip()
                lvl = grpLvl.rsplit(sep,1)[1].strip()
        else:
            grp = grpLvl.rsplit('-',1)[0].strip()
            #print(grp)
            lvl = grpLvl.rsplit('-',1)[1]
            #print(lvl)
            if (lvl == '00'):
                lvl = '0'
            else:
                lvl = grpLvl.rsplit('-',1)[1].lstrip('0').lstrip(sep).split('/')[0].strip()
        
        #print(grp + ' - ' + lvl)
        rateGrp = RateGroup(
            grp,
            lvl,       
            rateAgreements
        )
        rates.append(rateGrp)
        previousGrp = grp

    results = [rateGrp.to_dict() for rateGrp in rates]
    #print(results)
    with open('data/' + tbsId + '_' + secondary + '.json', 'w') as json_file:
        json.dump(results, json_file)
    return results

In [6]:
def getDurationClauses(tbsId):
    soup = getContentSoup(tbsId)
    
    durationSection = soup.select('h2:-soup-contains(": duration")') + soup.select('h3:-soup-contains(": duration")') + soup.select('h3:-soup-contains(": supervisory differential")') + soup.select('h2:-soup-contains(": no discrimination or sexual harassment")') + soup.select('h2:-soup-contains(": term of agreement")')

    if tbsId == '12':
        durationSection = soup.select('h3:-soup-contains(": term of agreement")') + soup.select('h3:-soup-contains(": maternity-related reassignment or leave")')
    elif tbsId == '14':
        durationSection = soup.select('h2:-soup-contains(": duration")') + soup.select('h2:-soup-contains(": agreement re-opener")')
    
    if len(durationSection) > 0:
        expireDate = durationSection[0].parent.select('p:-soup-contains("expire")')
    else:
        expireDate = []

    if len(expireDate) > 0:
        effectiveDate = durationSection[0].parent.select('p:-soup-contains("expire")')[0]
    else:
        effectiveDate = durationSection[0].parent.select('p:-soup-contains("agreement")')[0]


    effectiveDate.find('strong').decompose() #removes the subsection so it doesn't mess with the signed
    
    effectiveDate = parser.parse(effectiveDate.getText().strip(), fuzzy=True).date()
    if len(durationSection) == 1:
        signedDate = durationSection[0].parent.select('p:-soup-contains("Signed at", "effective on", "Signed in")')[-1]
    else:
        signedDate = durationSection[1].parent.select('p:-soup-contains("Signed at")')[0]
    
    for strongElems in signedDate.findAll('strong'): #removes the subsection so it doesn't mess with the signed
        strongElems.decompose()
    
    signedDate = parser.parse(signedDate.getText(), fuzzy=True).date()

    return {'effectiveDate': effectiveDate.strftime('%Y-%m-%d'), 'signedDate': signedDate.strftime('%Y-%m-%d')}

In [13]:
def getAgreements(agreementTable, steps, signingDate):
    rateAgreements = []
    for agreement in agreementTable:
        #print(agreement)
        rateStepsList = []
        amounts = agreement.findAll('td') #all the cells inside each row - since the row names are th we can get amts only with td
        for idx,amount in enumerate(amounts):
            if amount.find(' to ') == -1:
                newAmts = [amountgetText().replace(',', '')]
            else:
                newAmts = amount.getText().replace(',', '').split(' to ')

            step = re.search('Step.[0-9]*', steps[idx+1].getText())
            if step is None:
                agStep = steps[idx+1].getText().replace('Range', '1') #if its just one level and says range instead of a level
                agSteg = int(agStep)
            else:
                agStep = int(re.search('[0-9]',step[0])[0])

            agAmts = []
            for amt in newAmts:
                if amt.isdigit():
                    agAmts.append(int(amt))
            if len(agAmts) > 0:
                rateStep = RateSteps(agStep, agAmts)
                rateStepsList.append(rateStep)

        #Getting the date
        rateAgreementDate  = agreement.find('time')
        #print(rateAgreementDate)
        #Date not always available as an attr
        if rateAgreementDate is None:
            dateStr = agreement.find('th').contents[0].split(')')[1].lstrip(' Wage Adjustment - ')
            if dateStr.startswith('Restructure'):
                daysToRestructure = ''.join(re.findall('[0-9]*', dateStr)).strip()                
                signingDate = datetime.strptime(signingDate, '%Y-%m-%d')
                postRestrucDate = signingDate + timedelta(days=int(daysToRestructure))
                #print(postRestrucDate.date())
                rateAgreementDate = postRestrucDate.date().strftime('%Y-%m-%d')
            else:
                rateAgreementDate = parser.parse(dateStr).date()
                rateAgreementDate = rateAgreementDate.strftime('%Y-%m-%d')
        else:
            rateAgreementDate = rateAgreementDate.get('datetime')
        
        rateAgreement = RateAgreement(rateAgreementDate, rateStepsList)
        rateAgreements.append(rateAgreement)

    return rateAgreements

In [8]:
def getJSON(tbsId, appendixStr, sep, secondary):
    signingDate = getDurationClauses(tbsId)
    getDurationClauses(tbsId)
    rateTables = getRateTables(tbsId, appendixStr)
    return parseHTMLToJSON(tbsId, rateTables, sep, secondary, signingDate['signedDate'])

In [114]:
page_1 = getJSON('1', '**Appendix A', ':', '0')
page_2 = getJSON('2', '**Appendix A', ':', '0')
page_3 = getJSON('3', '**Appendix A', ':', '0')
page_4 = getJSON('4', '**Appendix A', '-', '0')
page_5 = getJSON('5', '**Appendix A', ':', '0')
page_6 = getJSON('6', '**Appendix A', ':', '0')
page_7 = getJSON('7', '** Appendix A', ':', '0')
page_9 = getJSON('9', '**Appendix B-1', ':', '0')
page_10 = getJSON('10', '** Appendix A', ':', '0')
page_11 = getJSON('11', '**Appendix A', ' - ', '0')
page_12 = getJSON('12', '**Appendix A', ':', '0')
page_15 = getJSON('15', '**Appendix A‑1', '-', '0')
page_16 = getJSON('16', '**Appendix A', ':', '0')
page_16 = getJSON('17', '**Appendix A', ':', '0')
page_18 = getJSON('18', '**Appendix A', ':', '0')
page_25 = getJSON('25', '**Appendix A', ':', '0')
page_25_1 = getJSON('25', '**Appendix A‑1', ':', '1') #we need to add "spec" to the model
page_26 = getJSON('26', '**Appendix A', ' - ', '0')
page_27 = getJSON('27', '**Appendix A', ':', '0')

In [10]:
#with open('data/combined/payscales.json', 'w') as json_file:
        #json.dump(combined_list, json_file)

In [11]:
#getJSON('14', '**Appendix A', ':') -> yeah this won't work... region specific
#getJSON('8', '**Appendix A', ' - ') -> yeah this ALSO won't work
#getJSON('13', '**Appendix A', ' - ') -> region specific